In [ ]:
import numpy as np
import glob
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from skimage.io import imread
from skimage.transform import resize
from skimage.feature import hog

# 1. Load and resize images
data_path = '/Users/rongrongqian/Desktop/AI modules/regression/UTKface/part1/*.jpg' #change the path
image_files = glob.glob(data_path)

ages = []
images_resized = []
# Standard size for images (128x128 pixels)
img_size = (128, 128)

for file in image_files[:100]:  # Use the first 100 images for speed
    age = int(os.path.basename(file).split('_')[0])  # Age is first part of filename
    img = imread(file)
    img_resized = resize(img, img_size, anti_aliasing=True)
    ages.append(age)
    images_resized.append(img_resized)

ages = np.array(ages)
images_resized = np.array(images_resized)

# 2. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    images_resized, ages, test_size=0.2, random_state=42
)

# 3. Extract HOG features
def extract_hog_features(images):
    hog_features = []  # List to store the HOG feature vector for each image
    for img in images:
        features, _ = hog(img, 
                          orientations=8,   # Number of orientation bins (e.g., 0°, 45°, ..., 315°)
                          pixels_per_cell=(16,16),  # Size of each cell in pixels (here 16x16)
                          cells_per_block=(2,2),   # Number of cells in each block (for normalization, here 2x2 cells)
                          visualize=True,   # Whether to return an image of the HOG (not used here, so we use '_')
                          channel_axis=-1)  # For color images: the color channel is the last dimension
        hog_features.append(features)  # Add the HOG feature vector for the current image to the list
    return np.array(hog_features)

X_train_hog = extract_hog_features(X_train)
X_test_hog = extract_hog_features(X_test)

# 4. Standardize HOG features to zero mean and unit variance
scaler = StandardScaler()
X_train_hog_scaled = scaler.fit_transform(X_train_hog)
X_test_hog_scaled = scaler.transform(X_test_hog)


# 5. Ordinary Least Squares (OLS) Regression - No regularization
hog_model_ols = LinearRegression().fit(X_train_hog_scaled, y_train)
hog_pred_ols = hog_model_ols.predict(X_test_hog_scaled)
mae_ols = mean_absolute_error(y_test, hog_pred_ols)
print("HOG Features OLS MAE:", mae_ols)

# 6. Ridge Regression with cross-validated alpha selection
# You can modify the 'alphas' list to search a wider or narrower range of values
hog_model_ridgecv = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)  # cv: number of folds
hog_model_ridgecv.fit(X_train_hog_scaled, y_train)
hog_pred_ridgecv = hog_model_ridgecv.predict(X_test_hog_scaled)
mae_ridgecv = mean_absolute_error(y_test, hog_pred_ridgecv)
print("HOG Features RidgeCV Best Alpha:", hog_model_ridgecv.alpha_)
print("HOG Features RidgeCV MAE:", mae_ridgecv)

# 7. Lasso Regression with cross-validated alpha selection
# You can adjust 'alphas' for the range of regularization strengths,
# and 'max_iter' if you see convergence warnings
hog_model_lassocv = LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10], cv=5, max_iter=5000)
hog_model_lassocv.fit(X_train_hog_scaled, y_train)
hog_pred_lassocv = hog_model_lassocv.predict(X_test_hog_scaled)
mae_lassocv = mean_absolute_error(y_test, hog_pred_lassocv)
print("HOG Features LassoCV Best Alpha:", hog_model_lassocv.alpha_)
print("HOG Features LassoCV MAE:", mae_lassocv)

HOG Features OLS MAE: 15.424845488258057
HOG Features RidgeCV Best Alpha: 100.0
HOG Features RidgeCV MAE: 15.564732183618872
HOG Features LassoCV Best Alpha: 10.0
HOG Features LassoCV MAE: 18.701067254268192
